In [1]:
import re
import pandas as pd
from pathlib import Path

In [2]:
run_dir = Path(r"../data\processed_data\run_2026-04-28_153116")

In [5]:
base_csv = Path(r"C:\Users\amcclary\Documents\GitHub\Transportation\TravelDemandModel\2026_Housing_EIS\Base\data\processed_data\SocioEcon_Summer.csv")

sum_vars = [
    'total_residential_units', 'total_occ_units',
    'occ_units_low_inc', 'occ_units_med_inc', 'occ_units_high_inc',
    'total_persons',
    'emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other',
]
emp_cols = ['emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other']

df_base = pd.read_csv(base_csv)
base_record = {v: int(df_base[v].sum()) for v in sum_vars}
base_record['total_jobs'] = int(df_base[emp_cols].sum().sum())

base_summary = pd.DataFrame({'Base': base_record}, index=sum_vars + ['total_jobs'])
base_summary.index.name = 'variable'
base_summary.to_csv('base_summary.csv')
print(f"Saved: base_summary.csv")
base_summary

Saved: base_summary.csv


,Base
variable,
total_residential_units,49950
total_occ_units,23655
occ_units_low_inc,9929
occ_units_med_inc,4838
occ_units_high_inc,8881
total_persons,54679
emp_retail,3711
emp_srvc,7503
emp_rec,2248


In [3]:
base_parcels_path = r"C:\Users\amcclary\Documents\GitHub\Transportation\TravelDemandModel\2026_Housing_EIS\Base\data\processed_data\parcel_spatial_joins.parquet"
df_parcels = pd.read_parquet(base_parcels_path)

MF_LANDUSE = {'Condominium', 'Condomunium Common Area'}

def classify_parcel(lu, units):
    if units > 1 :
        return 'MF'
    elif lu in MF_LANDUSE:
        return 'MF Condo'
    else:
        return 'SF'

df_parcels['parcel_type'] = df_parcels.apply(
    lambda r: classify_parcel(r['EXISTING_LANDUSE'], r['Residential_Units']), axis=1
)

# ── Main summary ──────────────────────────────────────────────────────────────
parcel_type_summary = (
    df_parcels.groupby('parcel_type')
    .agg(parcel_count=('APN', 'count'), residential_units=('Residential_Units', 'sum'))
    .reindex(['SF', 'MF', 'MF Condo'])
    .fillna(0)
    .astype(int)
)
parcel_type_summary.loc['Total'] = parcel_type_summary.sum()

# ── MF breakdown by existing landuse ─────────────────────────────────────────
mf_by_landuse = (
    df_parcels[df_parcels['parcel_type'] == 'MF']
    .groupby('EXISTING_LANDUSE')
    .agg(parcel_count=('APN', 'count'), residential_units=('Residential_Units', 'sum'))
    .sort_values('parcel_count', ascending=False)
)
mf_by_landuse.loc['Total'] = mf_by_landuse.sum()

parcel_type_summary.to_csv(run_dir / 'base_parcel_type_summary.csv')
mf_by_landuse.to_csv(run_dir / 'base_mf_by_landuse.csv')
print(f"Saved: {run_dir / 'base_parcel_type_summary.csv'}")
print(f"Saved: {run_dir / 'base_mf_by_landuse.csv'}")
print("\n── Parcel Type Summary ──")
display(parcel_type_summary)
print("\n── MF Parcels by Land Use ──")
display(mf_by_landuse)

Saved: ..\data\processed_data\run_2026-04-28_153116\base_parcel_type_summary.csv
Saved: ..\data\processed_data\run_2026-04-28_153116\base_mf_by_landuse.csv

── Parcel Type Summary ──


,parcel_count,residential_units
parcel_type,,
SF,53003,34601
MF,2138,8753
MF Condo,6135,6108
Total,61276,49462



── MF Parcels by Land Use ──


,parcel_count,residential_units
EXISTING_LANDUSE,,
Multi-Family Residential,1679,7248
Single Family Residential,308,675
,66,357
Commercial,32,179
Vacant,21,137
Tourist Accommodation,15,117
Condominium,7,17
Condominium Common Area,6,13
Open Space,3,7


In [24]:
sum_vars = [
    'total_residential_units', 'total_occ_units',
    'occ_units_low_inc', 'occ_units_med_inc', 'occ_units_high_inc',
    'total_persons',
    'emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other',
]
emp_cols = ['emp_retail', 'emp_srvc', 'emp_rec', 'emp_game', 'emp_other']

records = {}

for scenario_dir in sorted(d for d in run_dir.iterdir() if d.is_dir() and d.name != 'configs'):
    clean_name = re.sub(r'_\d{8}', '', scenario_dir.name)
    for year in [2035, 2050]:
        csv_path = scenario_dir / str(year) / 'SocioEcon_Summer.csv'
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        key = f"{clean_name}_{year}"
        records[key] = {v: int(df[v].sum()) for v in sum_vars}
        records[key]['total_jobs'] = int(df[emp_cols].sum().sum())

comparison = pd.DataFrame(records, index=sum_vars + ['total_jobs'])
comparison.index.name = 'variable'

# Diff each scenario's 2050 vs scenario 1's 2050 (first sorted scenario)
cols_2050 = [c for c in comparison.columns if c.endswith('_2050')]
if len(cols_2050) >= 2:
    baseline = cols_2050[0]
    for col in cols_2050[1:]:
        label = col[: -len('_2050')]
        comparison[f'diff_{label}_vs_alt1_2050'] = comparison[col] - comparison[baseline]

comparison.to_csv(run_dir / 'scenario_comparison.csv')
print(f"Saved: {run_dir / 'scenario_comparison.csv'}")
comparison

Saved: ..\data\processed_data\run_2026-04-28_153116\scenario_comparison.csv


,Alternative_1_2035,Alternative_1_2050,Alternative_2_2035,Alternative_2_2050,Alternative_3_2035,Alternative_3_2050,diff_Alternative_2_vs_alt1_2050,diff_Alternative_3_vs_alt1_2050
variable,,,,,,,,
total_residential_units,51062,54197,51469,55435,51062,54197,1238,0
total_occ_units,24687,26507,24997,27750,25019,27362,1243,855
occ_units_low_inc,10563,11441,10709,11650,10563,11441,209,0
occ_units_med_inc,4889,5142,4952,5607,5042,5556,465,414
occ_units_high_inc,9235,9924,9336,10493,9414,10365,569,441
total_persons,55588,57608,56281,60244,56340,59473,2636,1865
emp_retail,3778,3855,3778,3855,3778,3855,0,0
emp_srvc,7632,7777,7632,7777,7632,7777,0,0
emp_rec,2286,2318,2286,2318,2286,2318,0,0


In [25]:
parquet_path = run_dir / 'all_scenarios_parcels.parquet'
if not parquet_path.exists():
    print(f"No parquet found at {parquet_path}")
else:
    df = pd.read_parquet(parquet_path)
    df['SCENARIO'] = df['SCENARIO'].apply(lambda s: re.sub(r'_\d{8}', '', s))
    df['reason_clean'] = (
        df['FORECAST_REASON']
        .fillna('')
        .str.replace(' Infill', '', regex=False)
        .str.replace(' MF', '', regex=False)
        .str.replace(' SF', '', regex=False)
        .str.strip()
    )

    # ── Residential units by reason × scenario ────────────────────────────────
    reason_summary = (
        df.groupby(['SCENARIO', 'reason_clean'])['FORECASTED_RESIDENTIAL_UNITS']
        .sum()
        .unstack('SCENARIO')
        .fillna(0)
        .astype(int)
    )
    reason_summary.index.name = 'forecast_reason'
    reason_summary.to_csv(run_dir / 'units_by_reason.csv')
    print(f"Saved: {run_dir / 'units_by_reason.csv'}")

    # ── Residential + income units by reason × scenario (wide) ───────────────
    income_cols = {
        'residential':  'FORECASTED_RESIDENTIAL_UNITS',
        'low_inc':      'FORECASTED_RES_INCOME_LOW_UNITS',
        'med_inc':      'FORECASTED_RES_INCOME_MEDIUM_UNITS',
        'high_inc':     'FORECASTED_RES_INCOME_HIGH_UNITS',
    }
    parts = []
    for metric, col in income_cols.items():
        pivot = (
            df.groupby(['SCENARIO', 'reason_clean'])[col]
            .sum()
            .unstack('SCENARIO')
            .fillna(0)
            .astype(int)
        )
        pivot.columns = [f"{s}_{metric}" for s in pivot.columns]
        parts.append(pivot)

    income_summary = pd.concat(parts, axis=1).fillna(0).astype(int)
    # Interleave columns so each scenario's 4 metrics stay together
    scenarios = df['SCENARIO'].unique()
    ordered_cols = [f"{s}_{m}" for s in sorted(scenarios) for m in income_cols]
    income_summary = income_summary[[c for c in ordered_cols if c in income_summary.columns]]
    income_summary.index.name = 'forecast_reason'
    income_summary.to_csv(run_dir / 'units_by_reason_income.csv')
    print(f"Saved: {run_dir / 'units_by_reason_income.csv'}")
    income_summary

Saved: ..\data\processed_data\run_2026-04-28_153116\units_by_reason.csv
Saved: ..\data\processed_data\run_2026-04-28_153116\units_by_reason_income.csv


In [4]:
taz_vars = [
    'total_residential_units', 'total_occ_units',
    'occ_units_low_inc', 'occ_units_med_inc', 'occ_units_high_inc',
    'total_persons',
]

taz_comparison = None
scenario_names = []

for scenario_dir in sorted(d for d in run_dir.iterdir() if d.is_dir() and d.name != 'configs'):
    csv_path = scenario_dir / '2050' / 'SocioEcon_Summer.csv'
    if not csv_path.exists():
        continue
    clean_name = re.sub(r'_\d{8}', '', scenario_dir.name)
    scenario_names.append(clean_name)
    df = pd.read_csv(csv_path, usecols=['taz'] + taz_vars)
    df = df.rename(columns={v: f"{clean_name}_{v}" for v in taz_vars})
    taz_comparison = df if taz_comparison is None else taz_comparison.merge(df, on='taz', how='outer')

if taz_comparison is not None:
    # Diff between scenario 2 and scenario 3 for each variable
    if len(scenario_names) >= 3:
        s2, s3 = scenario_names[1], scenario_names[2]
        for v in taz_vars:
            c2, c3 = f"{s2}_{v}", f"{s3}_{v}"
            if c2 in taz_comparison.columns and c3 in taz_comparison.columns:
                taz_comparison[f"diff_{s3}_vs_{s2}_{v}"] = taz_comparison[c3] - taz_comparison[c2]

    taz_comparison = taz_comparison.sort_values('taz').reset_index(drop=True)
    taz_comparison.to_csv(run_dir / 'taz_comparison_2050.csv', index=False)
    print(f"Saved: {run_dir / 'taz_comparison_2050.csv'}")
    taz_comparison

Saved: ..\data\processed_data\run_2026-04-28_153116\taz_comparison_2050.csv


In [6]:
# import packages
import pandas as pd
import pathlib
from pathlib import Path
import os
import arcpy
from utils import *
import numpy as np
import pickle
# external connection packages
from sqlalchemy.engine import URL
from sqlalchemy import create_engine

# pandas options
pd.options.mode.copy_on_write = True
pd.options.mode.chained_assignment = None
pd.options.display.max_columns = 999
pd.options.display.max_rows    = 999

# my workspace 
workspace = r"C:\GIS\Scratch.gdb"
# current working directory
local_path = pathlib.Path().absolute()

# get bonus_condit
# set data path as a subfolder of the current working directory TravelDemandModel\2022\
#data_dir = local_path.parents[0] / 'data'
# folder to save processed data
out_dir  = local_path.parents[0] / 'data/processed_data/base_data'
# workspace gdb for stuff that doesnt work in memory
# gdb = os.path.join(local_path,'Workspace.gdb')
gdb = workspace
# set environement workspace to in memory 
arcpy.env.workspace = 'memory'
# # clear memory workspace
# arcpy.management.Delete('memory')

# overwrite true
arcpy.env.overwriteOutput = True
# Set spatial reference to NAD 1983 UTM Zone 10N
sr = arcpy.SpatialReference(26910)

# get parcels from the database
# network path to connection files
filePath = "F:/GIS/PARCELUPDATE/Workspace/"
# database file path 
sdeBase    = os.path.join(filePath, "Vector.sde")
sdeCollect = os.path.join(filePath, "Collection.sde")
sdeTabular = os.path.join(filePath, "Tabular.sde")
sdeEdit    = os.path.join(filePath, "Edit.sde")

# Pickle variables
# part 1 - spatial joins and new categorical fields
parcel_pickle_part1    = out_dir / 'attributed_parcels.geoparquet'


# columsn to list
initial_columns = [ 'APN',
                    'APO_ADDRESS',
                    'Residential_Units',
                    'TouristAccommodation_Units',
                    'CommercialFloorArea_SqFt',
                    'YEAR',
                    'JURISDICTION',
                    'COUNTY',
                    'OWNERSHIP_TYPE',
                    'COUNTY_LANDUSE_DESCRIPTION',
                    'EXISTING_LANDUSE',
                    'REGIONAL_LANDUSE',
                    'YEAR_BUILT',
                    'PLAN_ID',
                    'PLAN_NAME',
                    'ZONING_ID',
                    'ZONING_DESCRIPTION',
                    'TOWN_CENTER',
                    'LOCATION_TO_TOWNCENTER',
                    'TAZ',
                    'PARCEL_ACRES',
                    'PARCEL_SQFT',
                    'WITHIN_BONUSUNIT_BNDY',
                    'WITHIN_TRPA_BNDY',
                    'VHR']

In [7]:

# read in travel demand model base year parcel data - CHANGE With new name
parcel_base_tdm  = r"Base\data\processed_data\parcel_spatial_joins.parquet"
parcel_base_path = local_path.parents[1].joinpath(parcel_base_tdm)
# read in base year parcel data
sdf_parcel_base  = pd.read_parquet(parcel_base_path)
#output_gdb = r"C:\GIS\Scratch.gdb"

In [8]:
sdf_parcel_base['EXISTING_LANDUSE'].unique()

<StringArray>
[                         '',                'Open Space',
                    'Vacant', 'Single Family Residential',
               'Condominium',   'Condominium Common Area',
     'Tourist Accommodation',  'Multi-Family Residential',
                'Commercial',            'Public Service',
                'Recreation']
Length: 11, dtype: string